### Build dataset files

In [ ]:
import re
ids_dict = {line.strip().split('\t')[1]:re.sub(r'\[.*\]', '', line.strip().split('\t')[2]) for line in open('cjkvi-ids/ids.txt', 'r').readlines() if not line.startswith('#') and len(line.strip().split('\t')) > 1}

### Recursive IDS

In [ ]:
# vocab_ids_dict
def get_full_ids(c):
    seq = ids_dict.get(c, c)
    if len(seq) > 1 and c not in seq:
        return ''.join([get_full_ids(cc) for cc in seq])
    return c     

In [ ]:
ids_exp_dict = {k: get_full_ids(k) for k in ids_dict}

In [ ]:
# build vocab
vocab_ids = set()
for k, v in ids_exp_dict.items():
    vocab_ids.update(v)

In [ ]:
ids_dict_lvl2 = {line.strip().split('\t')[0]:  re.sub(r"(?<!#)\([^)]*\)", "", line.strip().split('\t')[1].split(';')[0]) for line in open('yb-ids/ids_lv2.txt', 'r').readlines() if len(line.strip().split('\t')) > 1}

In [ ]:
oov = [idx for idx in ids_dict_lvl2.keys() if ids_dict.get(idx) is None]
len(oov)

In [ ]:
# create filtered ids dict by extending vocab ids, 
# if the new char cannot map to this vocab, it is removed
ids_dict_filter = ids_dict.copy()
ids_dict_filter.update({k: k for k in vocab_ids})

In [ ]:
def get_full_ids_with_dict(c, ids_dict_from, ids_dict_to):
    seq = ids_dict_from.get(c, ' ') # use space for unmapped chars
    if len(seq) > 1 and c not in seq:
        return ''.join([get_full_ids_with_dict(cc, ids_dict_to, ids_dict_to) for cc in seq])
    return seq

In [ ]:
oov_ids = {idx: get_full_ids_with_dict(idx, ids_dict_lvl2, ids_dict_filter) for idx in oov}
oov_ids = {k: v for k, v in oov_ids.items() if ' ' not in v}
len(oov_ids)

In [ ]:
ids_exp_dict.update(oov_ids)

In [ ]:
# write to file
with open('ids_exp.txt', 'w') as f:
    f.writelines([f'{k}\t{v}\n' for k, v in ids_exp_dict.items()])

### Write vocab

In [ ]:
# write vocab ids full to file
with open('vocab_ids.txt', 'w') as f:
    f.write('\n'.join(sorted(vocab_ids)))

### Build encoder

In [ ]:
# load vocab ids
base_vocab = open('vocab_ids.txt', 'r').read().split('\n')
ids_dict = {line.strip().split('\t')[0]:line.strip().split('\t')[1] for line in open('ids_exp.txt', 'r').readlines()}

In [ ]:
from trie_search import Trie, TrieNode

class Vocab:
    def __init__(self, base_vocab, ids_dict):
        self.id2char = {i: c for i, c in enumerate(base_vocab)}
        self.char2id = {c: i for i, c in self.id2char.items()}
        self.size = len(base_vocab)
        self.ids_dict = ids_dict
        self.ids_dict_rev = {v: k for k, v in ids_dict.items()}

        self.trie = Trie()
        for k, v in ids_dict.items():
            self.trie.insert(self.encode(k))
        
    def __len__(self):
        return self.size
    
    def encode(self, c):
        return [self.char2id[c] for c in self.ids_dict[c]]

    def decode(self, ids):
        closest = self.trie.search_fuzzy(ids, max_distance=5)
        if len(closest) > 0:
            return self.ids_dict_rev[''.join([self.id2char[i] for i in closest[0][0]])]
        return None

In [ ]:
vocab = Vocab(base_vocab, ids_dict)

In [ ]:
vocab.decode(vocab.encode('閇'))